In [3]:
# Celula 1 - Setup Colab resiliente v2 (clone/zip + token opcional + validacao de import)
import os
import sys
import shutil
import subprocess
import urllib.request
import zipfile
import io
from pathlib import Path

SETUP_VERSION = "2026-04-29-v2"
print("SETUP_VERSION:", SETUP_VERSION)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = Path("/content/TCC")
BRANCH = "update"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()  # opcional para repo privado

def _run(cmd, *, capture=False):
    print("Executando:", " ".join(map(str, cmd)))
    return subprocess.run(cmd, text=True, capture_output=capture)

def _repo_http_base(repo_url: str) -> str:
    return repo_url[:-4] if repo_url.endswith(".git") else repo_url

def _repo_slug(repo_url: str) -> str:
    base = _repo_http_base(repo_url).rstrip("/")
    return "/".join(base.split("/")[-2:])  # owner/repo

def _auth_clone_url(repo_url: str) -> str:
    if not GITHUB_TOKEN:
        return repo_url
    base = _repo_http_base(repo_url)
    return base.replace("https://", f"https://{GITHUB_TOKEN}@") + ".git"

def _try_clone(repo_url: str, repo_dir: Path, branch: str | None):
    url = _auth_clone_url(repo_url)
    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [url, str(repo_dir)]
    proc = _run(cmd, capture=True)
    ok = proc.returncode == 0
    if not ok:
        print("[clone] falhou com codigo", proc.returncode)
        if proc.stderr:
            print(proc.stderr[-2500:])
    return ok

def _download_zip_codeload(repo_url: str, repo_dir: Path, branch: str):
    slug = _repo_slug(repo_url)
    zip_url = f"https://codeload.github.com/{slug}/zip/refs/heads/{branch}"
    print("Tentando ZIP:", zip_url)

    req = urllib.request.Request(zip_url)
    if GITHUB_TOKEN:
        req.add_header("Authorization", f"token {GITHUB_TOKEN}")

    data = urllib.request.urlopen(req).read()
    zf = zipfile.ZipFile(io.BytesIO(data))
    zf.extractall(repo_dir.parent)

    expected_prefix = f"{slug.split('/')[-1]}-{branch}"
    extracted = None
    for p in repo_dir.parent.iterdir():
        if p.is_dir() and p.name.startswith(expected_prefix):
            extracted = p
            break
    if extracted is None:
        raise RuntimeError(f"Pasta extraida nao encontrada para prefixo {expected_prefix}")
    extracted.rename(repo_dir)

def _obtain_repo(repo_url: str, repo_dir: Path, branch: str):
    errors = []

    if _try_clone(repo_url, repo_dir, branch):
        return
    errors.append(f"clone-branch:{branch}")

    if _try_clone(repo_url, repo_dir, None):
        return
    errors.append("clone-default")

    for b in [branch, "main", "master"]:
        if b in {e.replace('clone-branch:', '') for e in []}:
            pass
        try:
            _download_zip_codeload(repo_url, repo_dir, b)
            print(f"Repositorio obtido via ZIP da branch '{b}'.")
            return
        except Exception as exc:
            print(f"ZIP da branch '{b}' falhou: {exc}")
            errors.append(f"zip:{b}")

    raise RuntimeError("Nao foi possivel obter repositorio. Tentativas: " + ", ".join(errors))

def _ensure_importable(repo_dir: Path):
    src = repo_dir / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

    try:
        import nvs_benchmark  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_dir)], check=True)
        import nvs_benchmark  # noqa: F401

    probe = [
        sys.executable,
        "-c",
        "import nvs_benchmark,sys; print('nvs_benchmark OK em', sys.executable)",
    ]
    subprocess.run(probe, check=True)

if IN_COLAB:
    if REPO_DIR.exists():
        print("Removendo pasta existente:", REPO_DIR)
        shutil.rmtree(REPO_DIR)

    _obtain_repo(REPO_URL, REPO_DIR, BRANCH)

    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

    req = REPO_DIR / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)

    _ensure_importable(REPO_DIR)
    print("Setup concluido. Modulo nvs_benchmark importavel.")
else:
    print("Nao detectado Colab. Ajuste os caminhos para execucao local.")

SETUP_VERSION: 2026-04-29-v2
Nao detectado Colab. Ajuste os caminhos para execucao local.


In [ ]:
# Celula 2 - Parametros da matriz completa
RUN_ID = "colab_full_matrix"
PRESET = "quick"  # smoke, quick, preview, standard, full
STRICT_RESULTS = True
MIN_REQUIRED_PAIRS = 1
GENERATE_PDF = False

# Controle de escopo (None = todos os disponiveis)
ONLY_DATASETS = None  # ex.: ["blender_synthetic"]
ONLY_METHODS = None   # ex.: ["nerf_static", "gs_static"]

# Observabilidade e modo de execucao
VERBOSE_MODE = True
VERBOSE_HEARTBEAT_SECONDS = 30
RUN_MODE = "full"  # "full" | "quick_check"
QUICK_CHECK_PRESET = "smoke"
QUICK_CHECK_MAX_COMBOS = 4

# Regras de combinacao
APPLY_COMPATIBILITY_FILTER = True
SKIP_METHODS = []  # ex.: ["gs_static"]

# Caminhos base
ARTIFACTS_DIR = Path("./artifacts")
METRICS_DIR = ARTIFACTS_DIR / "metrics"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
LOG_DIR = Path("./logs")

# Snapshot consolidado final
SNAPSHOT_FILE = METRICS_DIR / f"{RUN_ID}.json"
REPORT_NAME = f"{RUN_ID}_report"

# Opcional: tentar instalar datasets via catalogo antes de rodar
RUN_DATASET_INSTALL = True

In [12]:
# Celula 3 - Validacoes de ambiente e utilitarios
import shutil
import traceback
import threading
import time
from glob import glob

import torch

REPO_DIR = Path.cwd()
print("Diretorio atual:", REPO_DIR)
print("Python:", sys.executable)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if IN_COLAB:
    usage = shutil.disk_usage("/content")
    free_gb = usage.free / (1024**3)
    print(f"Espaco livre em /content: {free_gb:.2f} GB")

def _stream_reader(pipe, sink, prefix: str):
    for line in iter(pipe.readline, ""):
        sink.append(line)
        print(f"{prefix}{line}", end="")
    pipe.close()

def run_cmd(cmd: list[str], check=True, stream=False, heartbeat_seconds=30):
    print("\n$", " ".join(cmd))

    if not stream:
        return subprocess.run(cmd, check=check, text=True, capture_output=True)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )

    stdout_lines = []
    stderr_lines = []

    t_out = threading.Thread(
        target=_stream_reader,
        args=(proc.stdout, stdout_lines, "[stdout] "),
        daemon=True,
    )
    t_err = threading.Thread(
        target=_stream_reader,
        args=(proc.stderr, stderr_lines, "[stderr] "),
        daemon=True,
    )
    t_out.start()
    t_err.start()

    last_heartbeat = time.time()
    while proc.poll() is None:
        now = time.time()
        if heartbeat_seconds and (now - last_heartbeat) >= heartbeat_seconds:
            elapsed = int(now - (last_heartbeat - heartbeat_seconds if last_heartbeat else now))
            print(f"[heartbeat] processo ativo... ({heartbeat_seconds}s desde ultimo heartbeat)")
            last_heartbeat = now
        time.sleep(1)

    t_out.join(timeout=2)
    t_err.join(timeout=2)

    result = subprocess.CompletedProcess(
        args=cmd,
        returncode=proc.returncode,
        stdout="".join(stdout_lines),
        stderr="".join(stderr_lines),
    )
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
    return result

def ensure_imports():
    src = REPO_DIR / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
    try:
        import nvs_benchmark  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
        import nvs_benchmark  # noqa: F401

ensure_imports()
print("Imports principais validados.")

Diretorio atual: c:\Users\Admin\Projetos\TCC\notebooks
Python: c:\Users\Admin\Projetos\TCC\venv\Scripts\python.exe
CUDA disponivel: False
Imports principais validados.


In [18]:
# Celula 4 - Instalacao cross-platform de datasets
import json
import platform
from types import SimpleNamespace

METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

OS_NAME = platform.system()  # 'Windows', 'Linux', 'Darwin'
print(f"Sistema Operacional: {OS_NAME}")


def _project_root() -> Path:
    candidates = [Path.cwd().resolve()]
    if "REPO_DIR" in globals():
        candidates.insert(0, Path(REPO_DIR).resolve())

    for start in candidates:
        current = start
        for _ in range(8):
            if (current / "configs" / "install_catalog.json").exists():
                return current
            if current.parent == current:
                break
            current = current.parent

    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto")


PROJECT_ROOT = _project_root()
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")


def _to_project_path(path_str: str) -> Path:
    p = Path(path_str)
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / p).resolve()


def _resolve_catalog_path() -> Path:
    candidate = PROJECT_ROOT / "configs" / "install_catalog.json"
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Nao foi possivel localizar configs/install_catalog.json")


def _exec_cross_platform(command: str) -> int:
    """Executa comando shell de forma cross-platform (PowerShell no Windows, sh no resto)."""
    if OS_NAME == "Windows":
        proc = subprocess.run(["powershell", "-NoProfile", "-Command", command], text=True)
    else:
        proc = subprocess.run(["sh", "-c", command], text=True)
    return proc.returncode


def _dataset_extract_root(dataset_name: str, target_path: str) -> Path:
    """Define raiz correta de extracao para cada dataset."""
    dataset_roots = {
        "blender_synthetic": DATA_ROOT / "blender_synthetic",
        "d_nerf": DATA_ROOT / "d_nerf",
        "mipnerf360": DATA_ROOT / "mipnerf360",
        "tanks_and_temples": DATA_ROOT / "tanks_and_temples",
        "custom": DATA_ROOT / "custom",
    }
    return dataset_roots.get(dataset_name, _to_project_path(target_path))


def _is_manual_dataset(command: str, url: str) -> bool:
    cmd = (command or "").lower()
    lower_url = (url or "").lower()
    if "baixe manualmente" in cmd or "registre" in cmd:
        return True
    if "tanksandtemples.org" in lower_url:
        return True
    if lower_url and not lower_url.endswith(".zip") and "github.com" in lower_url:
        return True
    return False


def _install_dataset_wget_curl(dataset_name: str, url: str, extract_root: Path) -> int:
    """Baixa dataset usando wget/curl e extrai no diretorio correto."""
    extract_root.mkdir(parents=True, exist_ok=True)
    tmp_zip = f"/tmp/{dataset_name}.zip"

    cmd_wget = (
        f"wget -q '{url}' -O '{tmp_zip}' "
        f"&& unzip -o -q '{tmp_zip}' -d '{extract_root}' "
        f"&& rm -f '{tmp_zip}'"
    )
    ret = subprocess.run(["sh", "-c", cmd_wget], capture_output=True).returncode
    if ret == 0:
        print(f"[ok] {dataset_name}: baixado com wget em {extract_root}")
        return 0

    cmd_curl = (
        f"curl -s -L '{url}' -o '{tmp_zip}' "
        f"&& unzip -o -q '{tmp_zip}' -d '{extract_root}' "
        f"&& rm -f '{tmp_zip}'"
    )
    ret = subprocess.run(["sh", "-c", cmd_curl], capture_output=True).returncode
    if ret == 0:
        print(f"[ok] {dataset_name}: baixado com curl em {extract_root}")
        return 0

    print(f"[error] {dataset_name}: wget e curl falharam")
    return ret


def _install_dataset(item: dict, execute: bool = False) -> str:
    """Instala um dataset adaptando o comando ao OS."""
    dataset_name = item.get("id", "unknown")
    target_path = item.get("path", "")
    url = item.get("url", "")
    command = item.get("command", "")

    target = _to_project_path(target_path)
    dataset_root = _dataset_extract_root(dataset_name, target_path)

    if target.exists() or dataset_root.exists():
        return f"[skip] {dataset_name}: ja existe ({dataset_root})"

    if not execute:
        return f"[plan] {dataset_name}: seria instalado em {dataset_root}"

    if _is_manual_dataset(command, url):
        dataset_root.mkdir(parents=True, exist_ok=True)
        return (
            f"[manual] {dataset_name}: download manual necessario. "
            f"Destino esperado: {dataset_root}. Fonte: {url}"
        )

    if "powershell" in command.lower() or "invoke-webrequest" in command.lower():
        if OS_NAME != "Windows":
            print(f"[info] {dataset_name}: comando Windows detectado, usando wget/curl")
            ret = _install_dataset_wget_curl(dataset_name, url, dataset_root)
            return f"[ok] {dataset_name}" if ret == 0 else f"[error] {dataset_name}"
        ret = _exec_cross_platform(command)
        return f"[ok] {dataset_name}" if ret == 0 else f"[error] {dataset_name}"

    ret = _exec_cross_platform(command)
    return f"[ok] {dataset_name}" if ret == 0 else f"[error] {dataset_name}"


# Carrega catalogo com fallback robusto
catalog_datasets = []
catalog_path = None
try:
    catalog_path = _resolve_catalog_path()
    from nvs_benchmark.install import load_install_catalog

    catalog = load_install_catalog(str(catalog_path))
    for ds in getattr(catalog, "datasets", []):
        catalog_datasets.append(
            {
                "id": getattr(ds, "item_id", None) or getattr(ds, "id", None),
                "path": getattr(ds, "path", ""),
                "url": getattr(ds, "url", ""),
                "command": getattr(ds, "command", ""),
                "label": getattr(ds, "label", getattr(ds, "item_id", "dataset")),
            }
        )
except Exception as e:
    print(f"Aviso ao carregar via nvs_benchmark.install: {e}")

if not catalog_datasets:
    if catalog_path is None:
        catalog_path = _resolve_catalog_path()
    raw = json.loads(catalog_path.read_text(encoding="utf-8"))
    for ds in raw.get("datasets", []):
        catalog_datasets.append(
            {
                "id": ds.get("id", ""),
                "path": ds.get("path", ""),
                "url": ds.get("url", ""),
                "command": ds.get("command", ""),
                "label": ds.get("label", ds.get("id", "dataset")),
            }
        )

catalog = SimpleNamespace(
    datasets=[
        SimpleNamespace(
            item_id=d["id"],
            path=d["path"],
            url=d["url"],
            command=d["command"],
            label=d["label"],
        )
        for d in catalog_datasets
    ]
)

print("")
print("=" * 70)
print(f"Catalogo: {catalog_path}")
print(f"Datasets Disponiveis ({len(catalog_datasets)}):")
print("=" * 70)
for ds in catalog_datasets:
    status = "[ok]" if _to_project_path(ds["path"]).exists() else "[need]"
    print(f"{status} {ds['label']:30s} -> {ds['path']}")
print("")
print("Para instalar manualmente, ajuste ONLY_DATASETS e execute a Celula 4.5 abaixo.")

Sistema Operacional: Windows
Project root: C:\Users\Admin\Projetos\TCC

Catalogo: C:\Users\Admin\Projetos\TCC\configs\install_catalog.json
Datasets Disponiveis (4):
[ok] Blender Synthetic (NeRF)       -> ./data/blender_synthetic/nerf_synthetic/lego
[need] D-NeRF Dynamic Dataset         -> ./data/d_nerf
[need] Mip-NeRF 360 (cenas reais unbounded) -> ./data/mipnerf360
[need] Tanks and Temples (grande escala) -> ./data/tanks_and_temples

Para instalar manualmente, ajuste ONLY_DATASETS e execute a Celula 4.5 abaixo.


In [19]:
# Celula 4.5 - Instalar datasets selecionados (cross-platform)
# Edite DATASETS_TO_INSTALL para escolher quais baixar
# Preenchido com todos os datasets listados em configs/install_catalog.json
DATASETS_TO_INSTALL = ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples']  # ex.: ['blender_synthetic', 'd_nerf']
EXECUTE_INSTALL = False  # Mude para True para executar de verdade

if not catalog_datasets:
    raise RuntimeError("catalog_datasets vazio. Rode a Celula 4 antes da 4.5.")

if not DATASETS_TO_INSTALL:
    print("Nenhum dataset selecionado em DATASETS_TO_INSTALL.")
    print("")
    print("Para instalar, edite DATASETS_TO_INSTALL com os nomes desejados, ex.:")
    print("  DATASETS_TO_INSTALL = ['blender_synthetic']")
    print("")
    print("Depois mude EXECUTE_INSTALL = True e rode a celula novamente.")
else:
    selected_to_install = [d for d in catalog_datasets if d['id'] in DATASETS_TO_INSTALL]

    print("")
    print(f"Instalacao de {len(selected_to_install)} dataset(s) (OS: {OS_NAME})")
    print("=" * 70)

    for ds in selected_to_install:
        msg = _install_dataset(ds, execute=EXECUTE_INSTALL)
        print(msg)

    print("=" * 70)
    if EXECUTE_INSTALL:
        print("Instalacao concluida. Rode a Celula 5 para descobrir datasets instalados.")
    else:
        print("Modo de planejamento. Mude EXECUTE_INSTALL = True para instalar de verdade.")


Instalacao de 4 dataset(s) (OS: Windows)
[skip] blender_synthetic: ja existe (C:\Users\Admin\Projetos\TCC\data\blender_synthetic)
[plan] d_nerf: seria instalado em C:\Users\Admin\Projetos\TCC\data\d_nerf
[plan] mipnerf360: seria instalado em C:\Users\Admin\Projetos\TCC\data\mipnerf360
[plan] tanks_and_temples: seria instalado em C:\Users\Admin\Projetos\TCC\data\tanks_and_temples
Modo de planejamento. Mude EXECUTE_INSTALL = True para instalar de verdade.


In [21]:
# Celula 5 - Descoberta de datasets e metodos (com diagnostico melhorado)
ensure_imports()

from nvs_benchmark.data import SUPPORTED_DATASETS
from nvs_benchmark.methods import build_registry_with_all_methods


def _project_root_for_discovery() -> Path:
    if "PROJECT_ROOT" in globals():
        return Path(PROJECT_ROOT)
    current = Path.cwd().resolve()
    for _ in range(8):
        if (current / "configs" / "install_catalog.json").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return Path.cwd().resolve()


DISCOVERY_ROOT = _project_root_for_discovery()
DATA_DIR = DISCOVERY_ROOT / "data"
MIPNERF360_SCENES = {"bicycle", "bonsai", "counter", "garden", "kitchen", "room", "stump"}


def choose_scene_root(dataset_name: str) -> Path | None:
    """Busca raiz de cena por dataset usando marcadores apropriados."""
    base_paths = {
        "blender_synthetic": [DATA_DIR / "blender_synthetic"],
        "d_nerf": [DATA_DIR / "d_nerf"],
        "mipnerf360": [DATA_DIR / "mipnerf360", DATA_DIR],
        "tanks_and_temples": [DATA_DIR / "tanks_and_temples"],
        "custom": [DATA_DIR / "custom"],
    }

    marker_files = {
        "blender_synthetic": ["transforms_train.json"],
        "d_nerf": ["transforms_train.json"],
        "mipnerf360": ["poses_bounds.npy"],
        "tanks_and_temples": ["transforms_train.json", "poses_bounds.npy"],
        "custom": ["transforms_train.json", "poses_bounds.npy"],
    }

    candidates = base_paths.get(dataset_name, [])
    markers = marker_files.get(dataset_name, ["transforms_train.json"])

    for base in candidates:
        if not base.exists():
            continue
        for marker in markers:
            for found_path in base.rglob(marker):
                parent = found_path.parent
                if dataset_name == "mipnerf360":
                    parent_name = parent.name.lower()
                    path_str = str(parent).lower()
                    valid_scene = parent_name in MIPNERF360_SCENES
                    inside_mip_root = "mipnerf360" in path_str or "360_v2" in path_str
                    if not (valid_scene and inside_mip_root):
                        continue
                return parent

    return None


all_datasets = list(SUPPORTED_DATASETS)
selected_datasets = ONLY_DATASETS if ONLY_DATASETS else all_datasets
selected_datasets = [d for d in selected_datasets if d in all_datasets]

registry = build_registry_with_all_methods()
if hasattr(registry, "list_ids"):
    all_methods = registry.list_ids()
elif hasattr(registry, "keys"):
    all_methods = list(registry.keys())
else:
    raise RuntimeError("Nao foi possivel listar metodos no registry.")

selected_methods = ONLY_METHODS if ONLY_METHODS else all_methods
selected_methods = [m for m in selected_methods if m in all_methods]

print("Datasets suportados:", all_datasets)
print("Datasets selecionados:", selected_datasets)
print("Metodos selecionados:", selected_methods)
print("")
print(f"Raiz de descoberta: {DISCOVERY_ROOT}")

print("=" * 70)
print("Diagnostico de Datasets:")
print("=" * 70)
for ds_name in selected_datasets:
    root = choose_scene_root(ds_name)
    if root:
        print(f"[ok] {ds_name:30s} encontrado em {root}")
    else:
        base = DATA_DIR / ds_name
        if base.exists():
            print(f"[x]  {ds_name:30s} NAO encontrado")
            print(f"    Pasta existe: {base}")
            print("    Procurando arquivos indice recursivamente...")
            for item in base.rglob("*"):
                if item.name in {"transforms_train.json", "poses_bounds.npy"}:
                    print(f"      ENCONTRADO: {item}")
        else:
            print(f"[x]  {ds_name:30s} NAO encontrado (pasta {base} nao existe)")
print("=" * 70)
print("")

if not selected_methods:
    raise RuntimeError("Nenhum metodo encontrado no registry. Verifique dependencias externas.")

Datasets suportados: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Datasets selecionados: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Metodos selecionados: ['gs_dynamic', 'gs_static', 'nerf_dynamic', 'nerf_static']

Raiz de descoberta: C:\Users\Admin\Projetos\TCC
Diagnostico de Datasets:
[ok] blender_synthetic              encontrado em C:\Users\Admin\Projetos\TCC\data\blender_synthetic\nerf_synthetic\lego
[x]  d_nerf                         NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\d_nerf nao existe)
[x]  mipnerf360                     NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\mipnerf360 nao existe)
[x]  tanks_and_temples              NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\tanks_and_temples nao existe)
[x]  custom                         NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\custom nao existe)



In [ ]:
# Celula 5.2 - Setup third_party repos (dependencies)
import io as io_module

third_party_repos = {
    "gaussian_splatting": {
        "url": "https://github.com/graphdeco-inria/gaussian-splatting.git",
        "branch": "main",
        "zip_url": "https://codeload.github.com/graphdeco-inria/gaussian-splatting/zip/refs/heads/main",
    },
    "d_nerf": {
        "url": "https://github.com/albertpumarola/D-NeRF.git",
        "branch": "main",
        "zip_url": "https://codeload.github.com/albertpumarola/D-NeRF/zip/refs/heads/main",
    },
    "nerf": {
        "url": "https://github.com/bmild/nerf.git",
        "branch": "master",
        "zip_url": "https://codeload.github.com/bmild/nerf/zip/refs/heads/master",
    },
}

third_party_dir = Path("./third_party")
third_party_dir.mkdir(exist_ok=True)

def clone_third_party_via_git(repo_name, url, branch, repo_path, retries=3):
    """Tenta clonar repo via git."""
    for attempt in range(1, retries + 1):
        print(f"  [git tentativa {attempt}/{retries}] ...", end=" ", flush=True)
        
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        
        cmd = [
            "git", "clone", "--depth", "1",
            "--branch", branch,
            url,
            str(repo_path)
        ]
        
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if proc.returncode == 0:
                print("OK")
                return True
            else:
                print(f"falhou (code={proc.returncode})")
        except subprocess.TimeoutExpired:
            print("timeout")
        except Exception as e:
            print(f"erro ({e})")
    
    return False

def clone_third_party_via_zip(repo_name, zip_url, repo_path, expected_prefix):
    """Tenta baixar repo via ZIP como fallback."""
    print(f"  [zip fallback] ...", end=" ", flush=True)
    
    try:
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        
        req = urllib.request.Request(zip_url)
        data = urllib.request.urlopen(req, timeout=60).read()
        
        zf = zipfile.ZipFile(io_module.BytesIO(data))
        zf.extractall(third_party_dir)
        
        # Procura pasta extraida e renomeia
        extracted = None
        for p in third_party_dir.iterdir():
            if p.is_dir() and p.name.startswith(expected_prefix):
                extracted = p
                break
        
        if extracted is None:
            print(f"falhou (pasta {expected_prefix}* nao encontrada)")
            return False
        
        extracted.rename(repo_path)
        print("OK")
        return True
    
    except Exception as e:
        print(f"falhou ({str(e)[:50]})")
        return False

print("=" * 70)
print("Verificando third_party repos:")
print("=" * 70)

for repo_name, repo_info in third_party_repos.items():
    repo_path = third_party_dir / repo_name
    
    # Check if already exists with .git
    if repo_path.exists() and (repo_path / ".git").exists():
        print(f"✓ {repo_name:25s} ja existe (git)")
        continue
    
    if repo_path.exists() and not (repo_path / ".git").exists():
        print(f"⚠ {repo_name:25s} existe mas sem .git (removendo...)")
        shutil.rmtree(repo_path, ignore_errors=True)
    
    print(f"⏳ {repo_name:25s}", end=" ")
    
    # Try git clone first
    if clone_third_party_via_git(
        repo_name,
        repo_info["url"],
        repo_info["branch"],
        repo_path
    ):
        print(f"✓ {repo_name:25s} clonado com git")
    # Fallback to ZIP
    elif clone_third_party_via_zip(
        repo_name,
        repo_info["zip_url"],
        repo_path,
        repo_name
    ):
        print(f"✓ {repo_name:25s} extraido via zip")
    else:
        print(f"✗ {repo_name:25s} FALHA permanente")
        print(f"    Tentativas: git clone + zip download")
        print(f"    Verifique conectividade ou firewall")

print("=" * 70)


In [ ]:
# Celula 5.1 - Diagnostico: Verificar third_party existente
third_party_path = Path("./third_party")

print("=" * 70)
print("Diagnostico de third_party/:")
print("=" * 70)

if third_party_path.exists():
    dirs = [d.name for d in third_party_path.iterdir() if d.is_dir()]
    print(f"Encontrado {len(dirs)} diretorios em ./third_party/:")
    for d in dirs:
        git_ok = "✓ git" if (third_party_path / d / ".git").exists() else "✗ sem git"
        print(f"  - {d:30s} ({git_ok})")
else:
    print("Diretorio ./third_party/ nao existe ainda.")

print("")
print("Se todas as dependencias ja estao presentes, pode pular a celula 5.2.")
print("Se faltarem, a celula 5.2 fara clone automaticamente.")
print("=" * 70)
print("")


In [ ]:
# Celula 5.5 - Diagnostico: Procurar transforms_train.json em todo ./data
import os

print("Busca completa por transforms_train.json em ./data/:")
print("=" * 70)

data_path = Path("./data")
if not data_path.exists():
    print("[error] Diretorio ./data nao existe")
else:
    found_any = False
    for item in data_path.rglob("transforms_train.json"):
        found_any = True
        print(f"✓ {item}")

    if not found_any:
        print("[nenhum] Nenhum transforms_train.json encontrado em ./data/")

    print("")
    print("Estrutura de ./data/ (ate 2 niveis):")
    print("=" * 70)
    for root, dirs, files in os.walk("./data", topdown=True):
        level = root.replace("./data", "").count(os.sep)
        if level > 2:
            dirs[:] = []
            continue

        indent = " " * (2 * level)
        folder_name = os.path.basename(root) or "data"
        print(f"{indent}{folder_name}/")

        subindent = " " * (2 * (level + 1))
        for file_name in sorted(files)[:10]:
            print(f"{subindent}{file_name}")
        if len(files) > 10:
            print(f"{subindent}... ({len(files) - 10} mais arquivos)")

    print("")
    print("Se ainda nao encontrar datasets, rode novamente Cell 4.5 com EXECUTE_INSTALL=True")

In [5]:
# Celula 5.6 - Mapear scene roots por dataset
from collections import defaultdict

try:
    dataset_ids = [d.item_id for d in catalog.datasets]
except Exception:
    try:
        from nvs_benchmark.install import load_install_catalog
        _catalog = load_install_catalog("./configs/install_catalog.json")
        dataset_ids = [d.item_id for d in _catalog.datasets]
    except Exception:
        dataset_ids = ["blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples", "custom"]

scene_roots = defaultdict(set)
all_transforms = sorted(Path("./data").rglob("transforms_train.json")) if Path("./data").exists() else []

for tf in all_transforms:
    parts = list(tf.parts)
    matched = None
    for ds in dataset_ids:
        if ds in parts:
            matched = ds
            break

    if matched is None:
        try:
            idx = parts.index("data")
            matched = parts[idx + 1] if idx + 1 < len(parts) else "unknown"
        except ValueError:
            matched = "unknown"

    scene_roots[matched].add(str(tf.parent))

print("=" * 70)
print("Mapa de scene roots em ./data")
print("=" * 70)
print(f"Total transforms_train.json encontrados: {len(all_transforms)}")
print("")

if not scene_roots:
    print("[vazio] Nenhum scene root encontrado.")
else:
    for ds in sorted(scene_roots.keys()):
        roots = sorted(scene_roots[ds])
        print(f"{ds:20s}: {len(roots)} scene(s)")
        for root in roots[:10]:
            print(f"  - {root}")
        if len(roots) > 10:
            print(f"  ... ({len(roots) - 10} adicionais)")
        print("")

Mapa de scene roots em ./data
Total transforms_train.json encontrados: 0

[vazio] Nenhum scene root encontrado.


In [6]:
# Celula 5.7 - Normalizar estruturas duplicadas (dry-run por padrao)
import json
import shutil
from datetime import datetime, timezone

EXECUTE_NORMALIZE = False
TARGET_DATASETS = DATASETS_TO_INSTALL if "DATASETS_TO_INSTALL" in globals() and DATASETS_TO_INSTALL else None

if TARGET_DATASETS is None:
    try:
        TARGET_DATASETS = [d.item_id for d in catalog.datasets]
    except Exception:
        TARGET_DATASETS = ["blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples", "custom"]


def find_duplicate_nested_dirs(dataset_root: Path):
    pairs = []
    if not dataset_root.exists():
        return pairs

    for parent in dataset_root.rglob("*"):
        if not parent.is_dir():
            continue
        inner = parent / parent.name
        if inner.is_dir():
            pairs.append((parent, inner))

    return pairs


def build_actions(parent: Path, inner: Path, dataset: str):
    actions = []
    for src in sorted(inner.iterdir(), key=lambda p: p.name):
        dst = parent / src.name
        actions.append(
            {
                "dataset": dataset,
                "parent": str(parent),
                "inner": str(inner),
                "src": str(src),
                "dst": str(dst),
                "conflict": dst.exists(),
            }
        )
    return actions


proposed_actions = []
all_pairs = []

for ds in TARGET_DATASETS:
    dataset_root = Path("./data") / ds
    pairs = find_duplicate_nested_dirs(dataset_root)
    for parent, inner in pairs:
        all_pairs.append((ds, parent, inner))
        proposed_actions.extend(build_actions(parent, inner, ds))

print("=" * 70)
print("Normalizacao de datasets: flatten X/X")
print("=" * 70)
print(f"Modo: {'EXECUTE' if EXECUTE_NORMALIZE else 'DRY-RUN'}")
print(f"Datasets alvo: {TARGET_DATASETS}")
print(f"Pares duplicados encontrados: {len(all_pairs)}")
print(f"Acoes propostas: {len(proposed_actions)}")
print("")

if not proposed_actions:
    print("Nenhuma acao necessaria.")
else:
    for i, action in enumerate(proposed_actions, 1):
        marker = "[CONFLICT]" if action["conflict"] else "[OK]"
        print(f"{i:03d}. {marker} {action['src']} -> {action['dst']}")

applied_actions = []
skipped_conflicts = []
errors = []

if EXECUTE_NORMALIZE and proposed_actions:
    for action in proposed_actions:
        src = Path(action["src"])
        dst = Path(action["dst"])

        if action["conflict"]:
            skipped_conflicts.append(action)
            continue

        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            applied_actions.append(action)
        except Exception as exc:
            action_err = dict(action)
            action_err["error"] = str(exc)
            errors.append(action_err)

    # Remove diretorios internos vazios apos move
    for _, _, inner in all_pairs:
        try:
            if inner.exists() and not any(inner.iterdir()):
                inner.rmdir()
        except Exception:
            pass

print("")
print("Resumo:")
print(f"- Aplicadas: {len(applied_actions)}")
print(f"- Conflitos ignorados: {len(skipped_conflicts)}")
print(f"- Erros: {len(errors)}")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
normalize_log_file = ARTIFACTS_DIR / "normalize_log.json"
normalize_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "execute": EXECUTE_NORMALIZE,
    "target_datasets": TARGET_DATASETS,
    "pairs_found": len(all_pairs),
    "proposed": proposed_actions,
    "applied": applied_actions,
    "skipped_conflicts": skipped_conflicts,
    "errors": errors,
}
normalize_log_file.write_text(json.dumps(normalize_log, indent=2), encoding="utf-8")
print(f"Log salvo em: {normalize_log_file}")

Normalizacao de datasets: flatten X/X
Modo: DRY-RUN
Datasets alvo: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Pares duplicados encontrados: 0
Acoes propostas: 0

Nenhuma acao necessaria.

Resumo:
- Aplicadas: 0
- Conflitos ignorados: 0
- Erros: 0
Log salvo em: artifacts\normalize_log.json


In [ ]:
# Celula 6 - Construir plano de execucao method x dataset
matrix_plan = []
dataset_found_count = 0

# Compatibilidade recomendada por tipo de cena
dataset_type = {
    "blender_synthetic": "static",
    "mipnerf360": "static",
    "tanks_and_temples": "static",
    "d_nerf": "dynamic",
    "custom": "any",
}
method_type = {
    "gs_static": "static",
    "nerf_static": "static",
    "gs_dynamic": "dynamic",
    "nerf_dynamic": "dynamic",
}

compat_skips = 0
for dataset_name in selected_datasets:
    root = choose_scene_root(dataset_name)
    if root is None:
        print(f"[SKIP] {dataset_name}: nao encontrado em ./data")
        continue

    dataset_found_count += 1
    ds_kind = dataset_type.get(dataset_name, "any")

    for method_name in selected_methods:
        mt_kind = method_type.get(method_name, "any")
        if APPLY_COMPATIBILITY_FILTER and ds_kind != "any" and mt_kind != "any" and ds_kind != mt_kind:
            compat_skips += 1
            continue

        matrix_plan.append({
            "dataset": dataset_name,
            "root": str(root),
            "method": method_name,
            "split": "train",
        })

matrix_plan_full = list(matrix_plan)

if RUN_MODE == "quick_check":
    matrix_plan = matrix_plan[:QUICK_CHECK_MAX_COMBOS]

print("")
print(f"Total de datasets encontrados: {dataset_found_count}/{len(selected_datasets)}")
print(f"Total de metodos disponiveis: {len(selected_methods)}")
print(f"Combinacoes apos filtro de compatibilidade: {len(matrix_plan_full)}")
if compat_skips:
    print(f"Combinacoes puladas por incompatibilidade: {compat_skips}")
print(f"Modo de execucao: {RUN_MODE}")
if RUN_MODE == "quick_check":
    print(f"Quick check: {len(matrix_plan)} combinacao(oes), preset={QUICK_CHECK_PRESET}")
else:
    print(f"Execucao full: {len(matrix_plan)} combinacao(oes), preset={PRESET}")

if "gs_dynamic" in selected_methods:
    print("[nota] gs_dynamic nesta base esta em modo stub (execucao muito rapida e metricas de placeholder).")
    print("[nota] Para treino real, prefira gs_static/nerf_* com dependencias third_party completas.")

print("")
for i, item in enumerate(matrix_plan[:10], 1):
    print(f"{i:02d}. {item['dataset']:20s} x {item['method']:15s} -> {item['root']}")
if len(matrix_plan) > 10:
    print("... (mostrando apenas as 10 primeiras)")

if not matrix_plan:
    print("")
    print("ERRO: Nenhuma combinacao valida encontrada.")
    print("")
    print("Opcoes:")
    print("1. Ajuste RUN_MODE para quick_check e QUICK_CHECK_MAX_COMBOS")
    print("2. Revise APPLY_COMPATIBILITY_FILTER")
    print("3. Verifique datasets com a Celula 5 e 5.5")
    print("4. Se necessario, use ONLY_DATASETS/ONLY_METHODS")
    raise RuntimeError("Nenhuma combinacao valida encontrada. Verifique os datasets em ./data.")

In [ ]:
# Celula 6.5 - Pre-check rapido (sem treino pesado)
print("=" * 70)
print("Pre-check rapido: methods-check + dataset-check")
print("=" * 70)

# 1) Checa metodos/dependencias
methods_check_cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "methods-check",
    "--output-dir", str(ARTIFACTS_DIR),
    "--log-dir", str(LOG_DIR),
]
methods_check = run_cmd(methods_check_cmd, check=False, stream=VERBOSE_MODE, heartbeat_seconds=VERBOSE_HEARTBEAT_SECONDS)
print(f"methods-check returncode={methods_check.returncode}")

# 2) Checa datasets encontrados no plano
roots_by_dataset = {}
for item in matrix_plan_full:
    roots_by_dataset[item["dataset"]] = item["root"]

for ds, root in roots_by_dataset.items():
    print("-" * 70)
    print(f"dataset-check: {ds} @ {root}")
    ds_cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "dataset-check",
        "--dataset", ds,
        "--root", root,
        "--split", "train",
    ]
    ds_res = run_cmd(ds_cmd, check=False, stream=VERBOSE_MODE, heartbeat_seconds=VERBOSE_HEARTBEAT_SECONDS)
    print(f"dataset-check[{ds}] returncode={ds_res.returncode}")

print("=" * 70)
print("Pre-check concluido.")
print("Dica: use RUN_MODE='quick_check' para validar pipeline completo em poucos pares.")

In [ ]:
# Celula 7 - Executar matriz completa (verbose e sem timeout)
run_results = []

effective_plan = [
    item for item in matrix_plan if item["method"] not in set(SKIP_METHODS)
]

if len(effective_plan) != len(matrix_plan):
    skipped = len(matrix_plan) - len(effective_plan)
    print(f"[info] {skipped} combinacao(oes) removidas por SKIP_METHODS={SKIP_METHODS}")

run_preset = QUICK_CHECK_PRESET if RUN_MODE == "quick_check" else PRESET
print(f"Preset efetivo desta rodada: {run_preset}")
print(f"Verbose: {VERBOSE_MODE} (heartbeat={VERBOSE_HEARTBEAT_SECONDS}s)")

for idx, item in enumerate(effective_plan, 1):
    ds = item["dataset"]
    method = item["method"]
    root = item["root"]
    pair_id = f"{ds}__{method}"
    pair_snapshot = METRICS_DIR / f"{RUN_ID}_{pair_id}.json"

    cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "method-run",
        "--method", method,
        "--dataset", ds,
        "--root", root,
        "--split", item["split"],
        "--preset", run_preset,
        "--output-dir", str(ARTIFACTS_DIR),
        "--log-dir", str(LOG_DIR),
        "--compute-metrics",
        "--snapshot-file", str(pair_snapshot),
        "--append-snapshot",
    ]
    if STRICT_RESULTS:
        cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

    print(f"\n[{idx}/{len(effective_plan)}] Rodando: {ds} x {method}")
    proc = run_cmd(
        cmd,
        check=False,
        stream=VERBOSE_MODE,
        heartbeat_seconds=VERBOSE_HEARTBEAT_SECONDS,
    )
    ok = proc.returncode == 0

    if not VERBOSE_MODE:
        if proc.stdout:
            print(proc.stdout[-1500:])
        if not ok and proc.stderr:
            print(proc.stderr[-2000:])

    run_results.append({
        "dataset": ds,
        "method": method,
        "root": root,
        "snapshot_file": str(pair_snapshot),
        "ok": ok,
        "returncode": proc.returncode,
    })

ok_count = sum(1 for r in run_results if r["ok"])
print(f"Concluido: {ok_count}/{len(run_results)} combinacoes com sucesso")

In [ ]:
# Celula 8 - Consolidar snapshots individuais em snapshot final
consolidated = {}

for item in run_results:
    snap = Path(item["snapshot_file"])
    if not item["ok"] or not snap.exists():
        continue
    try:
        payload = json.loads(snap.read_text(encoding="utf-8"))
        if isinstance(payload, dict):
            consolidated.update(payload)
    except Exception:
        print(f"Aviso: snapshot invalido ignorado: {snap}")

SNAPSHOT_FILE.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_FILE.write_text(json.dumps(consolidated, indent=2, ensure_ascii=False), encoding="utf-8")
print("Snapshot consolidado:", SNAPSHOT_FILE)
print("Entradas consolidadas:", len(consolidated))

In [ ]:
# Celula 9 - Resumo da execucao e diagnostico
from collections import Counter

status_counter = Counter("ok" if r["ok"] else "fail" for r in run_results)
print("Resumo:", dict(status_counter))

failed = [r for r in run_results if not r["ok"]]
if failed:
    print("\nCombinacoes com falha:")
    for f in failed[:20]:
        print(f"- {f['dataset']} x {f['method']} (code={f['returncode']})")
    if len(failed) > 20:
        print("... (mostrando apenas as 20 primeiras)")

if not consolidated:
    raise RuntimeError("Snapshot consolidado vazio. Nao ha resultados para gerar relatorio.")

In [ ]:
# Celula 10 - Gerar relatorio consolidado
report_cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", str(SNAPSHOT_FILE),
    "--output-dir", str(REPORTS_DIR),
    "--report-name", REPORT_NAME,
    "--log-dir", str(LOG_DIR),
]
if not GENERATE_PDF:
    report_cmd.append("--no-pdf")
if STRICT_RESULTS:
    report_cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"] )

proc = run_cmd(report_cmd, check=False)
if proc.stdout:
    print(proc.stdout[-2000:])
if proc.returncode != 0:
    if proc.stderr:
        print(proc.stderr[-2000:])
    raise RuntimeError("Falha ao gerar relatorio consolidado.")

REPORT_HTML = REPORTS_DIR / f"{REPORT_NAME}.html"
print("Relatorio HTML:", REPORT_HTML)

In [ ]:
# Celula 11 - Exibir relatorio no notebook
from IPython.display import IFrame, display

if REPORT_HTML.exists():
    display(IFrame(src=str(REPORT_HTML), width=1200, height=720))
else:
    print("Relatorio HTML nao encontrado:", REPORT_HTML)

In [ ]:
# Celula 12 - Compactar artefatos e baixar no Colab
import pathlib

archive_base = "/content/nvs_benchmark_full_matrix_artifacts"
archive_file = shutil.make_archive(archive_base, "zip", str(REPO_DIR), "artifacts")
print("Arquivo gerado:", archive_file)

if IN_COLAB and pathlib.Path(archive_file).exists():
    colab_files_mod = __import__("google.colab", fromlist=["files"])
    files = getattr(colab_files_mod, "files")
    files.download(archive_file)

In [ ]:
# Celula 13 - Backup opcional para Google Drive
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/NVS_Benchmark_Full_Matrix")

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")

    target_artifacts = DRIVE_OUTPUT_DIR / "artifacts"
    if target_artifacts.exists():
        shutil.rmtree(target_artifacts)
    shutil.copytree(ARTIFACTS_DIR, target_artifacts)
    print("Backup concluido em:", target_artifacts)
else:
    print("Backup no Drive desabilitado (USE_GOOGLE_DRIVE=False).")

In [ ]:
# Celula 14 - Resumo final
total = len(run_results)
ok = sum(1 for r in run_results if r["ok"])
fail = total - ok

print("=" * 70)
print("NVS Benchmark - Full Matrix (Colab)")
print("=" * 70)
print("Run ID:", RUN_ID)
print("Preset:", PRESET)
print(f"Combinacoes: {ok}/{total} sucesso, {fail} falha")
print("Snapshot consolidado:", SNAPSHOT_FILE)
print("Relatorio HTML:", REPORT_HTML)
print("Logs:", LOG_DIR)
print("=" * 70)

if fail > 0:
    print("Sugestao: execute novamente apenas combinacoes com falha usando ONLY_DATASETS/ONLY_METHODS.")